# 02b — Supervised Learning: Classification

Predict **categories** from input features.  
Spam/not-spam, disease/healthy, digit 0-9 — all classification.

```
Input Features ──► Model ──► Category
  (pixel values)            (digit = 7)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris, load_wine, load_digits, make_moons, make_circles
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

---
## 1 — What Is Classification?

| Regression | Classification |
|-----------|----------------|
| How much? | Which category? |
| Continuous output | Discrete output |
| Price = $425k | Class = "spam" |

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data[:, :2], iris.target

colors = ['#e74c3c', '#3498db', '#2ecc71']
plt.figure(figsize=(8, 5))
for i, name in enumerate(iris.target_names):
    mask = y_iris == i
    plt.scatter(X_iris[mask, 0], X_iris[mask, 1], c=colors[i], 
                label=name, edgecolors='k', linewidths=0.5, s=60)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Iris Dataset — 3 Classes to Separate')
plt.legend()
plt.show()

---
## 2 — Logistic Regression

Despite the name, it's a **classification** algorithm.  
Applies the **sigmoid function** to a linear model to output probabilities.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Output: probability between 0 and 1. If $p > 0.5$, predict class 1.

In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Decision threshold = 0.5')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.15, color='green', label='Predict class 1')
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.15, color='red', label='Predict class 0')
plt.xlabel('z = wx + b')
plt.ylabel('σ(z) = probability')
plt.title('Sigmoid Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def plot_decision_boundary(model, X, y, ax, title=''):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    cmap_bg = ListedColormap(['#ffcccc', '#ccccff', '#ccffcc'])
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for i in np.unique(y):
        mask = y == i
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i], edgecolors='k', linewidths=0.5, s=40)
    ax.set_title(title)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y_iris)

fig, ax = plt.subplots(figsize=(8, 6))
plot_decision_boundary(lr, X_scaled, y_iris, ax, f'Logistic Regression — Acc: {lr.score(X_scaled, y_iris):.3f}')
ax.set_xlabel(iris.feature_names[0] + ' (scaled)')
ax.set_ylabel(iris.feature_names[1] + ' (scaled)')
plt.tight_layout()
plt.show()

---
## 3 — K-Nearest Neighbors (KNN)

No training phase — just remember all data. To predict:  
1. Find the **k closest** training points  
2. **Vote** — majority class wins

Key hyperparameter: **k** (number of neighbors)  
- Small k (1): very flexible, overfits  
- Large k: smoother boundary, may underfit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k in zip(axes, [1, 7, 50]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_scaled, y_iris)
    acc = knn.score(X_scaled, y_iris)
    plot_decision_boundary(knn, X_scaled, y_iris, ax, f'KNN (k={k}) — Acc: {acc:.3f}')

plt.suptitle('k=1 overfits (jagged boundary) · Large k underfits (too smooth)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_iris, test_size=0.3, random_state=42)

k_range = range(1, 40)
train_acc = []
test_acc = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    train_acc.append(knn.score(X_tr, y_tr))
    test_acc.append(knn.score(X_te, y_te))

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), train_acc, 'b-', label='Train accuracy')
plt.plot(list(k_range), test_acc, 'r-', label='Test accuracy')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Accuracy')
plt.title('Finding the sweet spot for k')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 4 — Decision Trees

Learn a series of **if-else rules** from the data.  
Intuitive and interpretable — you can literally read the rules.

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_scaled, y_iris)

plt.figure(figsize=(14, 7))
plot_tree(dt, feature_names=iris.feature_names[:2], class_names=iris.target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree — Human-readable rules')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, depth in zip(axes, [1, 3, None]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_scaled, y_iris)
    label = f'Depth={depth}' if depth else 'No limit'
    plot_decision_boundary(dt, X_scaled, y_iris, ax, f'Decision Tree ({label}) — Acc: {dt.score(X_scaled, y_iris):.3f}')

plt.suptitle('Deeper trees = more complex boundaries = risk of overfitting', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5 — Random Forest

Instead of one tree → grow **many trees** on random subsets and **vote**.  
Why it works: individual trees overfit differently, voting cancels out their errors.

```
      Tree 1: "Class A"   Tree 2: "Class B"   Tree 3: "Class A"
                          ↓
                    Final vote: "Class A" (2 vs 1)
```

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target
X_wtr, X_wte, y_wtr, y_wte = train_test_split(X_w, y_w, test_size=0.3, random_state=42)

dt_wine = DecisionTreeClassifier(random_state=42).fit(X_wtr, y_wtr)
rf_wine = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True).fit(X_wtr, y_wtr)

print(f'Decision Tree  →  Test accuracy: {dt_wine.score(X_wte, y_wte):.3f}')
print(f'Random Forest  →  Test accuracy: {rf_wine.score(X_wte, y_wte):.3f}')
print(f'Random Forest  →  OOB score:     {rf_wine.oob_score_:.3f}')

In [ ]:
importances = rf_wine.feature_importances_
idx = np.argsort(importances)[-10:]

plt.figure(figsize=(8, 5))
plt.barh(range(len(idx)), importances[idx], color='steelblue', edgecolor='black')
plt.yticks(range(len(idx)), np.array(wine.feature_names)[idx])
plt.xlabel('Feature Importance')
plt.title('Random Forest — Top 10 Features')
plt.tight_layout()
plt.show()

---
## 6 — Support Vector Machines (SVM)

Find the **hyperplane** that separates classes with the **widest margin**.  
Points closest to the boundary are called **support vectors**.

**Kernel trick**: project data into higher dimensions where it *is* linearly separable.

In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, kernel, title in zip(axes, ['linear', 'rbf'],
    ['Linear Kernel — can\'t handle curves', 'RBF Kernel — handles non-linear boundaries']):
    svm = SVC(kernel=kernel, C=1.0)
    svm.fit(X_moons, y_moons)
    
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdBu', edgecolors='k', linewidths=0.5)
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=150, facecolors='none', edgecolors='gold', linewidths=2, label='Support vectors')
    ax.set_title(f'{title} — Acc: {svm.score(X_moons, y_moons):.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, kernel in zip(axes, ['linear', 'rbf']):
    svm = SVC(kernel=kernel)
    svm.fit(X_circles, y_circles)
    
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='RdBu', edgecolors='k', linewidths=0.5)
    ax.set_title(f'{kernel.upper()} kernel — Acc: {svm.score(X_circles, y_circles):.3f}')

plt.suptitle('Concentric circles — linear fails, RBF handles it', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7 — Model Comparison

Train all classifiers on the same dataset and compare. Fair fight.

In [ ]:
iris_full = load_iris()
X_full, y_full = iris_full.data, iris_full.target

scaler = StandardScaler()
X_full_s = scaler.fit_transform(X_full)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf'),
}

results = {}
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_full_s, y_full, cv=5, scoring='accuracy')
    results[name] = {'Mean Accuracy': scores.mean(), 'Std': scores.std()}

results_df = pd.DataFrame(results).T.round(4)
results_df.sort_values('Mean Accuracy', ascending=False)

In [ ]:
names = list(results.keys())
means = [results[n]['Mean Accuracy'] for n in names]
stds = [results[n]['Std'] for n in names]

plt.figure(figsize=(10, 5))
bars = plt.barh(names, means, xerr=stds, color='steelblue', edgecolor='black', capsize=5)
plt.xlabel('5-Fold CV Accuracy')
plt.title('Classifier Comparison on Iris Dataset')
plt.xlim(0.8, 1.0)

for bar, mean in zip(bars, means):
    plt.text(mean + 0.005, bar.get_y() + bar.get_height()/2, f'{mean:.3f}', va='center')

plt.tight_layout()
plt.show()

---
## 8 — Multiclass Classification

Most algorithms handle binary classification natively. For 3+ classes:

| Strategy | How it works |
|----------|-------------|
| **One-vs-Rest (OvR)** | Train k binary classifiers (one per class vs all others) |
| **One-vs-One (OvO)** | Train k(k-1)/2 classifiers (one per pair of classes) |

sklearn handles this automatically for most classifiers.

In [ ]:
digits = load_digits()
X_d, y_d = digits.data, digits.target

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, img, label in zip(axes.ravel(), digits.images[:10], digits.target[:10]):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('Digits Dataset — 8×8 pixel images, 10 classes', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
X_dtr, X_dte, y_dtr, y_dte = train_test_split(X_d, y_d, test_size=0.3, random_state=42)

scaler_d = StandardScaler()
X_dtr_s = scaler_d.fit_transform(X_dtr)
X_dte_s = scaler_d.transform(X_dte)

multi_models = {
    'Logistic (OvR)': LogisticRegression(max_iter=5000, multi_class='ovr'),
    'KNN':            KNeighborsClassifier(n_neighbors=3),
    'Random Forest':  RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':      SVC(kernel='rbf', gamma='scale'),
}

for name, model in multi_models.items():
    model.fit(X_dtr_s, y_dtr)
    acc = model.score(X_dte_s, y_dte)
    print(f'{name:20s} →  Test accuracy: {acc:.4f}')

In [ ]:
import seaborn as sns

best = multi_models['SVM (RBF)']
y_pred_d = best.predict(X_dte_s)

cm = confusion_matrix(y_dte, y_pred_d)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=digits.target_names, yticklabels=digits.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — SVM on Digits')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_dte, y_pred_d, target_names=[str(i) for i in range(10)]))

---
## 9 — When to Use Which Algorithm?

| Algorithm | Strengths | Weaknesses | Best for |
|-----------|-----------|------------|----------|
| Logistic Regression | Fast, interpretable, good baseline | Only linear boundaries | Binary/multi with linear separation |
| KNN | Simple, no training | Slow at prediction, curse of dimensionality | Small datasets, quick baseline |
| Decision Tree | Interpretable, handles mixed features | Overfits easily | When you need explainability |
| Random Forest | Robust, rarely overfits, feature importance | Slow to train, black box | General-purpose, tabular data |
| SVM | Effective in high dimensions, kernel trick | Slow on large data, need scaling | Medium datasets, non-linear boundaries |

**Rule of thumb**: Start with Logistic Regression (baseline) → Random Forest (strong default) → tune from there.

---
## Key Takeaways

| Concept | Remember |
|---------|----------|
| Logistic Regression | Sigmoid → probabilities → threshold → class |
| KNN | No training, just voting. k controls complexity. |
| Decision Trees | If-else rules. Depth controls overfitting. |
| Random Forest | Many trees voting. Hard to beat on tabular data. |
| SVM | Maximum margin. Kernel trick for non-linear. |
| Decision boundaries | Visualize them! They reveal model behavior. |

**Next**: Unsupervised learning — what happens when you have no labels? →